# Quick Start with AutoDDG

This notebook demonstrates how to generate dataset descriptions, expand them for search, and evaluate their quality using **AutoDDG**.

AutoDDG supports both **API-based** (OpenAI) and **local LLM** (transformers) modes.

---

## 1. Imports

In [ ]:
import pandas as pd
from openai import OpenAI

from autoddg import AutoDDG, GPTEvaluator
from autoddg.utils import get_sample

## 2. Initialization

Choose one of the following options:

### Option A: Using OpenAI API (Recommended for quick start)

In [ ]:
# Option A: OpenAI API
my_api_key = "YOUR_OPENAI_API_KEY"  # Replace with your key
client = OpenAI(api_key=my_api_key)
model_name = "gpt-4o-mini"

# Initialize AutoDDG with API client
auto_ddg = AutoDDG(client=client, model_name=model_name)

### Option B: Using Local LLM (Qwen, Llama, etc.)

For local LLM support, install the optional dependencies first:
```bash
pip install autoddg[local-llm]
# or
pip install transformers torch
```


In [ ]:
# Option B: Local LLM (uncomment to use)
# import torch
# 
# # Initialize AutoDDG with local LLM
# auto_ddg = AutoDDG(
#     client=None,
#     model_name="Qwen/Qwen2.5-7B-Instruct",  # or any HuggingFace model
#     use_local_llm=True,
#     local_llm_device="cuda",  # or "cpu" if no GPU
#     local_llm_dtype="bfloat16",  # or "float16", "float32"
# )


## 3. Load Dataset and Prepare Context

Here we sample rows, profile the dataset, extract semantic information, and generate a short topic.

In [ ]:
# Load dataset
csv_file = "clark_dataset.csv"
title = "Renal Cell Carcinoma"
original_description = (
    "This study reports a large-scale proteogenomic analysis of ccRCC to discern the functional impact "
    "of genomic alterations and provides evidence for rational treatment selection stemming from ccRCC pathobiology"
)
csv_df = pd.read_csv(csv_file)

# Sample rows
sample_df, dataset_sample = get_sample(csv_df, sample_size=100)

# Generate profiles
basic_profile, structural_profile = auto_ddg.profile_dataframe(csv_df)
semantic_profile_details = auto_ddg.analyze_semantics(sample_df)
semantic_profile = "\n".join(
    section for section in [structural_profile, semantic_profile_details] if section
)

# Generate topic
data_topic = auto_ddg.generate_topic(
    title=title,
    original_description=original_description,
    dataset_sample=dataset_sample,
)

## 4. Generate Descriptions

We create both a **general dataset description** and a **search-focused description**.

In [ ]:
# General description
prompt, description = auto_ddg.describe_dataset(
    dataset_sample=dataset_sample,
    dataset_profile=basic_profile,
    use_profile=True,
    semantic_profile=semantic_profile,
    use_semantic_profile=True,
    data_topic=data_topic,
    use_topic=True,
)

# Search-focused description
search_prompt, search_focused_description = auto_ddg.expand_description_for_search(
    description=description,
    topic=data_topic,
)

###  General Description

In [ ]:
description

###  Search-Focused Description

In [ ]:
search_focused_description

## 5. Evaluate Quality

Finally, we use the evaluator to score both descriptions.

In [ ]:
# Attach evaluator (requires OpenAI API key)
# Note: Evaluation currently requires API access
try:
    auto_ddg.set_evaluator(GPTEvaluator(gpt4_api_key=my_api_key))
    
    # Score descriptions
    general_score = auto_ddg.evaluate_description(description)
    search_score = auto_ddg.evaluate_description(search_focused_description)
    
    print("Score of the general description:", general_score)
    print("Score of the search-focused description:", search_score)
except Exception as e:
    print(f"Evaluation skipped: {e}")
    print("Note: Evaluation requires OpenAI API access")